# 03 — ARMA Mean Model Selection

This notebook selects the optimal ARMA(p,q) mean model for BTC log returns
using ACF/PACF analysis and information criteria (AIC/BIC).

**Pipeline step 3 of 7.**

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_processed_data
from src.diagnostics import plot_acf_pacf

%matplotlib inline

## 3.1 Load Returns

In [ ]:
df = load_processed_data()
returns = df["log_return"].dropna()
print(f"Sample size: {len(returns)}")

## 3.2 ACF and PACF of Log Returns

In [ ]:
fig = plot_acf_pacf(returns, lags=30, title_prefix="Log Returns")
plt.savefig("../results/figures/acf_pacf_returns.png", dpi=150)
plt.show()

## 3.3 Grid Search: ARMA Order Selection

Fit ARMA(p,q) models for p,q ∈ {0, 1, 2} and select by AIC and BIC.

In [ ]:
results = []
for p, q in itertools.product(range(3), range(3)):
    try:
        model = ARIMA(returns, order=(p, 0, q)).fit()
        results.append({"p": p, "q": q, "AIC": model.aic, "BIC": model.bic})
    except Exception:
        pass

arma_df = pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)
arma_df.to_csv("../results/tables/arma_selection.csv", index=False)
print(arma_df.head(10))

## 3.4 Best ARMA Model

In [ ]:
best = arma_df.iloc[0]
print(f"Best ARMA model by AIC: ARMA({int(best.p)},{int(best.q)})")
print(f"  AIC = {best.AIC:.4f}, BIC = {best.BIC:.4f}")